In [54]:
import pandas as pd
import numpy as np

In [55]:
df = pd.read_csv(
    r"D:\Election Data Project\data\processed\Elections_Clean.csv"
)

#### Convert Date Column

In [56]:
df.head()

,Country,country_type,election_year,election_date,largest_electorate,largest_winner,second_electorate,second_winner,third_electorate,third_winner,national_winner
0,United States,Developed,1964,1964-11-03,California,Lyndon B. Johnson,Texas,Lyndon B. Johnson,Florida,Lyndon B. Johnson,Lyndon B. Johnson
1,United States,Developed,1968,1968-11-05,California,Richard Nixon,Texas,Richard Nixon,Florida,Lyndon B. Johnson,Richard Nixon
2,United States,Developed,1972,1972-11-07,California,Richard Nixon,Texas,Richard Nixon,Florida,Richard Nixon,Richard Nixon
3,United States,Developed,1976,1976-11-02,California,Jimmy Carter,Texas,Jimmy Carter,Florida,Jimmy Carter,Jimmy Carter
4,United States,Developed,1980,1980-11-04,California,Ronald Reagan,Texas,Ronald Reagan,Florida,Ronald Reagan,Ronald Reagan


print("Dataset Shape:", df.shape)

In [57]:
df["election_date"] = pd.to_datetime(
    df["election_date"]
)

In [58]:
df.dtypes

Country                       object
country_type                  object
election_year                  int64
election_date         datetime64[ns]
largest_electorate            object
largest_winner                object
second_electorate             object
second_winner                 object
third_electorate              object
third_winner                  object
national_winner               object
dtype: object

#### Sort Dataset

In [59]:
df = df.sort_values(
    ["Country", "election_year"]
).reset_index(drop=True)

### FEATURE ENGINEERING

In [60]:
def get_majority_winner(row):

    winners = [
        row["largest_winner"],
        row["second_winner"],
        row["third_winner"]
    ]

    winner_count = pd.Series(winners).value_counts()

    return winner_count.index[0]


df["majority_winner"] = df.apply(
    get_majority_winner,
    axis=1
)

In [61]:
df[
[
"largest_winner",
"second_winner",
"third_winner",
"majority_winner"
]
].head()

,largest_winner,second_winner,third_winner,majority_winner
0,Fernando Collor,Fernando Collor,Fernando Collor,Fernando Collor
1,Fernando Henrique Cardoso,Fernando Henrique Cardoso,Fernando Henrique Cardoso,Fernando Henrique Cardoso
2,Fernando Henrique Cardoso,Fernando Henrique Cardoso,Fernando Henrique Cardoso,Fernando Henrique Cardoso
3,Jos‚ Serra,Lula,Lula,Lula
4,Geraldo Alckmin,Lula,Lula,Lula


### Prediction Correct
##### Did top 3 largest electoral regions predict the national winner?

In [62]:
df["prediction_correct"] = (
    df["majority_winner"]
    ==
    df["national_winner"]
)

In [63]:
df["prediction_correct"].value_counts()

prediction_correct
True     51
False     6
Name: count, dtype: int64

### Match Count
##### Counts how many largest regions selected the national winner.

In [64]:
df["match_count"] = (

    (df["largest_winner"] 
     == df["national_winner"]).astype(int)

    +

    (df["second_winner"] 
     == df["national_winner"]).astype(int)

    +

    (df["third_winner"] 
     == df["national_winner"]).astype(int)

)

In [65]:
df["match_count"].value_counts()

match_count
3    27
2    24
1     4
0     2
Name: count, dtype: int64

### Agreement Score

In [66]:
df["agreement_score"] = (
    df["match_count"] / 3
)

### Prediction Confidence

In [67]:
def confidence_level(score):

    if score == 1:
        return "Very High"

    elif score >= 0.67:
        return "Medium"

    else:
        return "Low"


df["prediction_confidence"] = (
    df["agreement_score"]
    .apply(confidence_level)
)

In [68]:
df["prediction_confidence"].value_counts()

prediction_confidence
Low          30
Very High    27
Name: count, dtype: int64

### Top 3 Region Sweep
##### Did one candidate win all 3 largest regions?

In [70]:
df["top3_sweep"] = (
    (df["largest_winner"] == df["second_winner"]) &
    (df["second_winner"] == df["third_winner"])
)


df[
[
"largest_winner",
"second_winner",
"third_winner",
"top3_sweep"
]
].head()

,largest_winner,second_winner,third_winner,top3_sweep
0,Fernando Collor,Fernando Collor,Fernando Collor,True
1,Fernando Henrique Cardoso,Fernando Henrique Cardoso,Fernando Henrique Cardoso,True
2,Fernando Henrique Cardoso,Fernando Henrique Cardoso,Fernando Henrique Cardoso,True
3,Jos‚ Serra,Lula,Lula,False
4,Geraldo Alckmin,Lula,Lula,False


### Regional Winner Diversity
##### Measures how many different candidates won the top 3 regions.

In [71]:
df["regional_winner_diversity"] = (

    df[
    [
    "largest_winner",
    "second_winner",
    "third_winner"
    ]
    ]
    .nunique(axis=1)

)

### Election Decade and Years Since Previous Election

In [72]:
df["election_decade"] = (

    (df["election_year"] // 10) * 10

)

In [73]:
df["years_since_previous_election"] = (

    df.groupby("Country")
    ["election_year"]
    .diff()

)

### Country Election Count

In [74]:
df["country_total_elections"] = (

    df.groupby("Country")
    ["Country"]
    .transform("count")

)

### Country Type Encoding

In [75]:
df["country_type_encoded"] = (

    df["country_type"]
    .map(
        {
        "Developed":1,
        "Developing":0
        }
    )

)

### Prediction Error Category

In [76]:
def error_category(row):

    if row["prediction_correct"]:
        return "Correct Prediction"

    elif row["match_count"] == 2:
        return "Close Failure"

    else:
        return "Major Failure"



df["prediction_error_category"] = (
    df.apply(error_category, axis=1)
)

### Candidate Regional Dominance

In [77]:
df["majority_winner_region_count"] = (

    (df["largest_winner"] 
     == df["majority_winner"]).astype(int)

    +

    (df["second_winner"] 
     == df["majority_winner"]).astype(int)

    +

    (df["third_winner"] 
     == df["majority_winner"]).astype(int)

)

In [78]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57 entries, 0 to 56
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Country                        57 non-null     object        
 1   country_type                   57 non-null     object        
 2   election_year                  57 non-null     int64         
 3   election_date                  57 non-null     datetime64[ns]
 4   largest_electorate             57 non-null     object        
 5   largest_winner                 57 non-null     object        
 6   second_electorate              57 non-null     object        
 7   second_winner                  57 non-null     object        
 8   third_electorate               57 non-null     object        
 9   third_winner                   57 non-null     object        
 10  national_winner                57 non-null     object        
 11  majority_winner      

In [79]:
df.describe()

,election_year,election_date,match_count,agreement_score,regional_winner_diversity,election_decade,years_since_previous_election,country_total_elections,country_type_encoded,majority_winner_region_count
count,57.000000,57,57.000000,57.000000,57.000000,57.000000,51.000000,57.000000,57.000000,57.000000
mean,2001.456140,2002-02-10 02:56:50.526315776,2.333333,0.777778,1.491228,1997.017544,4.607843,10.649123,0.596491,2.508772
min,1964.000000,1964-11-03 00:00:00,0.000000,0.000000,1.000000,1960.000000,4.000000,5.000000,0.000000,2.000000
25%,1991.000000,1991-01-13 00:00:00,2.000000,0.666667,1.000000,1990.000000,4.000000,9.000000,0.000000,2.000000
50%,2004.000000,2004-09-20 00:00:00,2.000000,0.666667,1.000000,2000.000000,5.000000,9.000000,1.000000,3.000000
75%,2014.000000,2014-10-26 00:00:00,3.000000,1.000000,2.000000,2010.000000,5.000000,16.000000,1.000000,3.000000
max,2024.000000,2024-11-05 00:00:00,3.000000,1.000000,2.000000,2020.000000,6.000000,16.000000,1.000000,3.000000
std,15.559873,NaN,0.763763,0.254588,0.504367,15.919991,0.634931,3.603031,0.494962,0.504367


In [80]:
df.to_csv(
    r"D:\Election Data Project\data\processed\Elections_Feature_Engineered.csv",
    index=False,
    encoding="utf-8"
)